In [16]:
import os
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
import matplotlib.pyplot as plt

In [17]:
# -------------------------------------------
# 1. Define transforms for training, validation, and test sets
# -------------------------------------------

# ResNet expects images of size 224x224 and normalized the same way it was trained
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),           # Resize images to 224x224
    transforms.RandomHorizontalFlip(),       # Augmentation: randomly flip horizontally
    transforms.ToTensor(),                   # Convert image to PyTorch tensor
    transforms.Normalize(                     # Normalize to ImageNet stats
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -------------------------------------------
# 2. Load datasets from folders
# -------------------------------------------
train_dataset = datasets.ImageFolder("data/train", transform=train_transforms)
val_dataset   = datasets.ImageFolder("data/val",   transform=val_transforms)
test_dataset  = datasets.ImageFolder("data/test",  transform=val_transforms)

# -------------------------------------------
# 3. Create DataLoaders
# -------------------------------------------
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

print("Classes:", train_dataset.classes)


Classes: ['Cat', 'Dog']


In [18]:
# -------------------------------------------
# 1. Load pretrained ResNet18
# -------------------------------------------
resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)  # Load ResNet18 with ImageNet weights

# -------------------------------------------
# 2. Freeze the convolutional base
# -------------------------------------------
for param in resnet.parameters():
    param.requires_grad = False  # Freeze all layers so we only train the classifier

# -------------------------------------------
# 3. Replace the final fully connected layer
# -------------------------------------------
num_features = resnet.fc.in_features  # Number of input features to final layer
resnet.fc = nn.Linear(num_features, 2)  # 2 classes: Cat and Dog

# -------------------------------------------
# 4. Move model to device (CPU or GPU)
# -------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
resnet = resnet.to(device)

print(resnet)  # Check model architecture

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
# -------------------------------------------
# 1. Loss function
# -------------------------------------------
criterion = nn.CrossEntropyLoss()  # Standard for multi-class classification

# -------------------------------------------
# 2. Optimizer
# -------------------------------------------
optimizer = optim.Adam(resnet.fc.parameters(), lr=0.001)  
# Only train the final layer (fc) since the rest is frozen

# -------------------------------------------
# 3. Training loop
# -------------------------------------------
epochs = 5  # You can increase later
for epoch in range(epochs):
    resnet.train()
    running_loss = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()           # Clear previous gradients
        outputs = resnet(images)       # Forward pass
        loss = criterion(outputs, labels)  # Compute loss
        loss.backward()                # Backpropagate
        optimizer.step()               # Update weights
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

# -------------------------------------------
# 4. Validation accuracy
# -------------------------------------------
resnet.eval()
correct = 0
total = 0

with torch.no_grad():  # No gradients needed for evaluation
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

val_acc = correct / total
print(f"Validation Accuracy: {val_acc:.2f}")

In [ ]:
# -------------------------------------------
# Save the trained model
# -------------------------------------------
torch.save(resnet.state_dict(), "cats_dogs_resnet.pth")
print("Model saved as cats_dogs_resnet.pth")